In [1]:
import pandas as pd
df = pd.read_csv('online_retail_II.csv', encoding='ISO-8859-1')

In [2]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


In [3]:
df = df.dropna(subset=['Customer ID'])

df['Invoice'] = df['Invoice'].astype(str)
df = df[~df['Invoice'].str.startswith('C')]

df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='mixed')
df['Total_Sales'] = df['Quantity'] * df['Price']

df = df[~df['Invoice'].str.contains('###')]

In [4]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Total_Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [5]:
from lifetimes.utils import summary_data_from_transaction_data

rfm = summary_data_from_transaction_data(
    df,
    customer_id_col='Customer ID',
    datetime_col='InvoiceDate',
    monetary_value_col='Total_Sales',
    observation_period_end=df['InvoiceDate'].max(),
    freq='D'  
)

In [6]:
rfm.head()

,frequency,recency,T,monetary_value
Customer ID,,,,
12346.0,0.0,0.0,325.0,0.000000
12347.0,6.0,365.0,367.0,599.701667
12348.0,3.0,283.0,358.0,301.480000
12349.0,0.0,0.0,18.0,0.000000
12350.0,0.0,0.0,310.0,0.000000


In [7]:
rfm_filtered = rfm[(rfm['frequency'] > 0) & (rfm['monetary_value'] > 0)]

print(f"Total customers: {len(rfm)}")
print(f"Repeat customers ready for modeling: {len(rfm_filtered)}")

Total customers: 4338
Repeat customers ready for modeling: 2790


In [8]:
rfm_filtered

,frequency,recency,T,monetary_value
Customer ID,,,,
12347.0,6.0,365.0,367.0,599.701667
12348.0,3.0,283.0,358.0,301.480000
12352.0,6.0,260.0,296.0,368.256667
12356.0,2.0,303.0,325.0,269.905000
12358.0,1.0,149.0,150.0,683.200000
...,...,...,...,...
18272.0,5.0,244.0,246.0,487.752000
18273.0,2.0,255.0,257.0,76.500000
18282.0,1.0,119.0,126.0,77.840000


In [9]:
from lifetimes import BetaGeoFitter

bgf = BetaGeoFitter(penalizer_coef=0.01)

bgf.fit(rfm_filtered['frequency'], rfm_filtered['recency'], rfm_filtered['T'])

print("Model trained successfully!")

Model trained successfully!


In [10]:
rfm_filtered['prob_alive'] = bgf.conditional_probability_alive(
    rfm_filtered['frequency'], 
    rfm_filtered['recency'], 
    rfm_filtered['T']
)

rfm_filtered['churn_risk'] = 1 - rfm_filtered['prob_alive']

rfm_filtered['churn_risk_tier'] = pd.cut(
    rfm_filtered['churn_risk'],
    bins=[0.0, 0.35, 0.70, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Churn Risk'],
    include_lowest=True
)

In [11]:
t_days = 90

rfm_filtered['predicted_purchases_90d'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    t_days,
    rfm_filtered['frequency'],
    rfm_filtered['recency'],
    rfm_filtered['T']
)

In [12]:
rfm_filtered[['frequency', 'recency', 'T', 'prob_alive', 'churn_risk', 'churn_risk_tier', 'predicted_purchases_90d']].sort_values(
    by='predicted_purchases_90d', ascending=False
).head()

,frequency,recency,T,prob_alive,churn_risk,churn_risk_tier,predicted_purchases_90d
Customer ID,,,,,,,
14911.0,131.0,372.0,373.0,0.999933,0.000067,Low Risk,25.763594
12748.0,112.0,373.0,373.0,0.999942,0.000058,Low Risk,22.072148
17841.0,111.0,372.0,373.0,0.999925,0.000075,Low Risk,21.877483
15311.0,89.0,373.0,373.0,0.999926,0.000074,Low Risk,17.603036
14606.0,88.0,372.0,373.0,0.999909,0.000091,Low Risk,17.408449


In [13]:
corr = rfm_filtered[['frequency', 'monetary_value']].corr().iloc[0, 1]
print(f"Correlation between frequency and monetary value: {corr:.3f}")

Correlation between frequency and monetary value: 0.016


In [14]:
from lifetimes import GammaGammaFitter

ggf = GammaGammaFitter(penalizer_coef=0.01)

ggf.fit(rfm_filtered['frequency'], rfm_filtered['monetary_value'])

print("Gamma-Gamma model trained successfully")

Gamma-Gamma model trained successfully


In [15]:
rfm_filtered['expected_avg_spend'] = ggf.conditional_expected_average_profit(
    rfm_filtered['frequency'],
    rfm_filtered['monetary_value']
)

In [16]:
rfm_filtered['clv_3_months'] = ggf.customer_lifetime_value(
    bgf, 
    rfm_filtered['frequency'],
    rfm_filtered['recency'],
    rfm_filtered['T'],
    rfm_filtered['monetary_value'],
    time=3,             
    discount_rate=0.01,   
    freq='D'              
)

In [17]:
rfm_filtered[[
    'frequency', 
    'monetary_value', 
    'prob_alive', 
    'expected_avg_spend', 
    'predicted_purchases_90d', 
    'clv_3_months'
]].sort_values(by='clv_3_months', ascending=False).head()

,frequency,monetary_value,prob_alive,expected_avg_spend,predicted_purchases_90d,clv_3_months
Customer ID,,,,,,
16446.0,1.0,168469.600000,0.953825,204762.443214,0.755212,151598.713591
14646.0,44.0,6366.705909,0.999833,6392.536964,9.237900,57892.102131
18102.0,25.0,9349.477200,0.999731,9416.375954,5.234978,48324.950576
17450.0,26.0,7404.690385,0.999579,7455.651066,5.430960,39694.925161
14096.0,16.0,4071.434375,0.999377,4117.267488,8.280162,33421.212146


In [38]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features_to_cluster = rfm_filtered[['churn_risk', 'clv_3_months']].copy()

features_to_cluster['log_clv'] = np.log1p(features_to_cluster['clv_3_months'])

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_to_cluster[['churn_risk', 'log_clv']])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_filtered['cluster_id'] = kmeans.fit_predict(scaled_features)

print(rfm_filtered['cluster_id'].value_counts())

cluster_id
2    1292
0     700
3     619
1     179
Name: count, dtype: int64


In [40]:
cluster_summary = rfm_filtered.groupby('cluster_id')[['churn_risk', 'clv_3_months', 'predicted_purchases_90d']].mean()
print(cluster_summary)

            churn_risk  clv_3_months  predicted_purchases_90d
cluster_id                                                   
0             0.069584    163.799099                 0.875521
1             0.389688    133.837437                 0.417647
2             0.019333    462.723513                 1.418892
3             0.011255   2467.216723                 2.728451


In [42]:
median_clv = rfm_filtered['clv_3_months'].median()
median_churn = rfm_filtered['churn_risk'].median()

def assign_segment(row):
    if row['clv_3_months'] >= median_clv and row['churn_risk'] < median_churn:
        return 'Champions'
    elif row['clv_3_months'] >= median_clv and row['churn_risk'] >= median_churn:
        return 'At-Risk High Value'
    elif row['clv_3_months'] < median_clv and row['churn_risk'] < median_churn:
        return 'Steady Low-Spenders'
    else:
        return 'Hibernating / Lost'

rfm_filtered['Customer_Segment'] = rfm_filtered.apply(assign_segment, axis=1)

In [44]:
print(rfm_filtered['Customer_Segment'].value_counts())

Customer_Segment
Champions              972
Hibernating / Lost     972
Steady Low-Spenders    423
At-Risk High Value     423
Name: count, dtype: int64


In [46]:
final_data = rfm_filtered.reset_index()

final_data.to_csv('customer_clv_segments.csv', index=False)
print("File successfully saved as customer_clv_segments.csv!")

File successfully saved as customer_clv_segments.csv!
